# Neural CSSR: Interactive Experiments

This notebook provides simple terminal commands to experiment with the complete Neural CSSR pipeline:

1. **Generate datasets** using `pysm_generator.py`
2. **Train models** using `train.py` 
3. **Extract FSMs** using `cssr_enhanced_extractor.py`
4. **Analyze results** with visualization scripts

Simply run the commands in your terminal or execute the cells below!

## Available Machines

Choose from these finite state machines:

| Machine | States | Complexity | Description |
|---------|--------|------------|-------------|
| `biased_coin` | 1 | Trivial | Simple biased coin (70% ones) |
| `golden_mean` | 2 | Simple | Classic 2-state baseline |
| `alternating` | 2 | Simple | Alternating 0-1 pattern |
| `distinct_3_state` | 3 | Low | Perfect recovery baseline |
| `unifilar_3_state` | 3 | Low | Overlapping probabilities |
| `distinct_4_state` | 4 | Medium | Moderate complexity |
| `distinct_6_state` | 6 | Medium-High | High separability |
| `seven_state_human` | 7 | High | Complex structure |

## Step 1: Generate Dataset

Pick a machine and generate a dataset:

In [ ]:
# Choose your machine (edit the machine name below)
MACHINE = "golden_mean"  # Change this to any machine from the table above
LENGTH = 80000           # Dataset length 
SEED = 42               # For reproducibility

# Create dedicated notebook folder for outputs
NOTEBOOK_FOLDER = f"notebook_experiments/{MACHINE}"

print(f"Generating {MACHINE} dataset with {LENGTH:,} symbols...")
print(f"All outputs will be saved in: {NOTEBOOK_FOLDER}/")
print(f"\nCommand to run:")
print(f"python pysm_generator.py --machine {MACHINE} --length {LENGTH} --output {NOTEBOOK_FOLDER}/data --seed {SEED}")

In [ ]:
# Execute the dataset generation
!python pysm_generator.py --machine {MACHINE} --length {LENGTH} --output {NOTEBOOK_FOLDER}/data --seed {SEED}

## Step 2: Create Training Configuration

Create a YAML configuration file for training:

In [ ]:
import yaml
from pathlib import Path

# Create configuration based on machine complexity
complexity_map = {
    'biased_coin': {'d_model': 32, 'n_layers': 2, 'n_heads': 2, 'epochs': 8},
    'golden_mean': {'d_model': 32, 'n_layers': 2, 'n_heads': 2, 'epochs': 10},
    'alternating': {'d_model': 32, 'n_layers': 2, 'n_heads': 2, 'epochs': 10},
    'distinct_3_state': {'d_model': 48, 'n_layers': 2, 'n_heads': 3, 'epochs': 10},
    'unifilar_3_state': {'d_model': 48, 'n_layers': 2, 'n_heads': 3, 'epochs': 12},
    'distinct_4_state': {'d_model': 64, 'n_layers': 3, 'n_heads': 4, 'epochs': 12},
    'distinct_6_state': {'d_model': 64, 'n_layers': 3, 'n_heads': 4, 'epochs': 15},
    'seven_state_human': {'d_model': 64, 'n_layers': 3, 'n_heads': 4, 'epochs': 15}
}

# Get config for selected machine
config_params = complexity_map.get(MACHINE, complexity_map['golden_mean'])

# Create configuration
config = {
    'experiment': {
        'name': f'{MACHINE}_notebook_experiment',
        'output_dir': f'{NOTEBOOK_FOLDER}/checkpoints'
    },
    'model': {
        'type': 'sliding_window',
        'vocab_size': 3,
        'd_model': config_params['d_model'],
        'n_layers': config_params['n_layers'], 
        'n_heads': config_params['n_heads'],
        'dropout': 0.1,
        'window_size': 25
    },
    'training': {
        'epochs': config_params['epochs'],
        'batch_size': 32,
        'learning_rate': 1e-3,
        'weight_decay': 0.01,
        'gradient_clipping': 1.0
    },
    'data': {
        'train_path': f'{NOTEBOOK_FOLDER}/data/{MACHINE}/{MACHINE}.dat',
        'chunk_size': 25,
        'sliding_window': True,
        'test_split': 0.2
    }
}

# Save configuration in notebook folder
config_dir = Path(f'{NOTEBOOK_FOLDER}/configs')
config_dir.mkdir(parents=True, exist_ok=True)
config_file = config_dir / f'{MACHINE}_config.yaml'

with open(config_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, indent=2)

print(f"Created configuration: {config_file}")
print(f"\nModel: {config_params['d_model']}-dim, {config_params['n_layers']} layers, {config_params['n_heads']} heads")
print(f"Training: {config_params['epochs']} epochs with sliding window")

# Display the config
print(f"\nConfiguration contents:")
print(yaml.dump(config, default_flow_style=False, indent=2))

## Step 3: Train Model

Train a sliding window transformer:

In [ ]:
print(f"Training {MACHINE} model...")
print(f"\nCommand to run:")
print(f"python train.py --config {NOTEBOOK_FOLDER}/configs/{MACHINE}_config.yaml")
print(f"\nThis will:")
print(f"• Load data from {NOTEBOOK_FOLDER}/data/{MACHINE}/{MACHINE}.dat")
print(f"• Train sliding window transformer for {config_params['epochs']} epochs")
print(f"• Save checkpoints to {NOTEBOOK_FOLDER}/checkpoints/")
print(f"• Use sliding window training (25x more data)")

In [ ]:
# Execute training
!python train.py --config {NOTEBOOK_FOLDER}/configs/{MACHINE}_config.yaml

## Step 4: Extract FSM

Extract finite state machine using CSSR-enhanced approach:

In [ ]:
# Set extraction parameters based on complexity
extraction_params = {
    'biased_coin': {'max_suffix_length': 4, 'max_sequences': 300, 'significance': 0.01},
    'golden_mean': {'max_suffix_length': 6, 'max_sequences': 500, 'significance': 0.01},
    'alternating': {'max_suffix_length': 6, 'max_sequences': 500, 'significance': 0.01},
    'distinct_3_state': {'max_suffix_length': 8, 'max_sequences': 750, 'significance': 0.001},
    'unifilar_3_state': {'max_suffix_length': 8, 'max_sequences': 750, 'significance': 0.001},
    'distinct_4_state': {'max_suffix_length': 10, 'max_sequences': 1000, 'significance': 0.001},
    'distinct_6_state': {'max_suffix_length': 10, 'max_sequences': 1000, 'significance': 0.001},
    'seven_state_human': {'max_suffix_length': 10, 'max_sequences': 1000, 'significance': 0.001}
}

params = extraction_params.get(MACHINE, extraction_params['golden_mean'])

print(f"Extracting FSM from {MACHINE} model...")
print(f"\nCommand to run:")
print(f"python cssr_enhanced_extractor.py \\")
print(f"  --checkpoint {NOTEBOOK_FOLDER}/checkpoints/best.pt \\")
print(f"  --data {NOTEBOOK_FOLDER}/data/{MACHINE}/{MACHINE}.dat \\")
print(f"  --output {NOTEBOOK_FOLDER}/results \\")
print(f"  --max-sequences {params['max_sequences']} \\")
print(f"  --max-suffix-length {params['max_suffix_length']} \\")
print(f"  --significance {params['significance']} \\")
print(f"  --use-information-theoretic-threshold")

print(f"\nExtraction parameters:")
print(f"• Max suffix length: {params['max_suffix_length']}")
print(f"• Max sequences: {params['max_sequences']}")
print(f"• Significance level: {params['significance']}")
print(f"• Information-theoretic threshold: ENABLED 🧠")
print(f"\nℹ️  Information-theoretic threshold:")
print(f"   Uses mutual information between neural distances and future distributions")
print(f"   Automatically finds optimal neural threshold (instead of fixed 5.0)")
print(f"   More principled approach for neural-classical fusion")

In [ ]:
# Execute FSM extraction
!python cssr_enhanced_extractor.py \
  --checkpoint {NOTEBOOK_FOLDER}/checkpoints/best.pt \
  --data {NOTEBOOK_FOLDER}/data/{MACHINE}/{MACHINE}.dat \
  --output {NOTEBOOK_FOLDER}/results \
  --max-sequences {params['max_sequences']} \
  --max-suffix-length {params['max_suffix_length']} \
  --significance {params['significance']} \
  --use-information-theoretic-threshold

## Step 5: Analyze Results

Analyze the extracted FSM:

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Load extraction results
results_file = Path(f'{NOTEBOOK_FOLDER}/results/cssr_enhanced_results.json')

if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    
    # Extract key information
    num_states = results['model_info']['num_causal_states']
    num_suffixes = results['model_info']['num_suffixes_processed']
    epsilon_machine = results['epsilon_machine']
    state_info = epsilon_machine.get('causal_state_info', {})
    
    # Get ground truth from our machine definitions
    ground_truth_states = {
        'biased_coin': 1, 'golden_mean': 2, 'alternating': 2,
        'distinct_3_state': 3, 'unifilar_3_state': 3, 'distinct_4_state': 4,
        'distinct_6_state': 6, 'seven_state_human': 7
    }
    
    expected_states = ground_truth_states.get(MACHINE, 'Unknown')
    
    print(f"🔍 EXTRACTION RESULTS FOR {MACHINE.upper()}")
    print("=" * 50)
    print(f"States extracted: {num_states}")
    print(f"Expected states: {expected_states}")
    print(f"Suffixes processed: {num_suffixes}")
    
    if isinstance(expected_states, int):
        recovery_ratio = num_states / expected_states
        if recovery_ratio <= 1.2:
            quality = "EXCELLENT ✅"
        elif recovery_ratio <= 2.0:
            quality = "GOOD ✅"
        else:
            quality = "OVER-SEGMENTED ⚠️"
        print(f"Recovery ratio: {recovery_ratio:.2f} ({quality})")
    
    print(f"\n📊 STATE ANALYSIS:")
    
    if state_info:
        # Analyze emission patterns
        emission_data = []
        for state_id, info in sorted(state_info.items()):
            future_probs = info.get('future_probabilities', {})
            p_0 = future_probs.get('0', 0.0)
            p_1 = future_probs.get('1', 0.0)
            count = info.get('count', 0)
            size = info.get('size', 0)
            
            emission_data.append((state_id, p_0, p_1, count, size))
            print(f"{state_id}: P(0)={p_0:.3f}, P(1)={p_1:.3f}, count={count}, suffixes={size}")
        
        # Create visualization
        if len(emission_data) > 1:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            
            # Emission patterns
            states = [data[0] for data in emission_data]
            p_0_vals = [data[1] for data in emission_data]
            p_1_vals = [data[2] for data in emission_data]
            
            x = np.arange(len(states))
            width = 0.35
            
            ax1.bar(x - width/2, p_0_vals, width, label='P(0)', alpha=0.8)
            ax1.bar(x + width/2, p_1_vals, width, label='P(1)', alpha=0.8)
            ax1.set_ylabel('Probability')
            ax1.set_title(f'Emission Patterns ({MACHINE})')
            ax1.set_xticks(x)
            ax1.set_xticklabels([s.replace('CS_', 'S') for s in states], rotation=45)
            ax1.legend()
            ax1.grid(axis='y', alpha=0.3)
            
            # State usage
            counts = [data[3] for data in emission_data]
            ax2.bar(range(len(states)), counts, alpha=0.8, color='orange')
            ax2.set_ylabel('Usage Count')
            ax2.set_title('State Usage Frequency')
            ax2.set_xticks(range(len(states)))
            ax2.set_xticklabels([s.replace('CS_', 'S') for s in states], rotation=45)
            ax2.grid(axis='y', alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        
        # Calculate separability
        if len(p_0_vals) > 1:
            separability_scores = []
            for i in range(len(p_0_vals)):
                for j in range(i+1, len(p_0_vals)):
                    diff = abs(p_0_vals[i] - p_0_vals[j])
                    separability_scores.append(diff)
            
            avg_sep = np.mean(separability_scores) if separability_scores else 0
            sep_quality = 'HIGH' if avg_sep > 0.3 else 'MODERATE' if avg_sep > 0.1 else 'LOW'
            print(f"\nSeparability: {avg_sep:.3f} ({sep_quality})")
    
    print(f"\n✅ Analysis complete! Results saved in: {results_file}")
    
else:
    print(f"❌ Results file not found: {results_file}")
    print("Make sure FSM extraction completed successfully.")

## Quick Experiments

Try these quick experiments by changing the `MACHINE` variable in the first cell:

### Simple Experiments (< 5 minutes each)
- `golden_mean`: 2-state baseline, perfect for learning the pipeline
- `distinct_3_state`: 3-state machine, demonstrates perfect recovery
- `biased_coin`: 1-state trivial case

### Complex Experiments (5-15 minutes each)  
- `seven_state_human`: 7-state complex machine, shows advanced capabilities
- `distinct_6_state`: 6-state with high separability
- `unifilar_3_state`: 3-state with overlapping probabilities

# Set extraction parameters based on complexity
extraction_params = {
    'biased_coin': {'max_suffix_length': 4, 'max_sequences': 300, 'significance': 0.01},
    'golden_mean': {'max_suffix_length': 6, 'max_sequences': 500, 'significance': 0.01},
    'alternating': {'max_suffix_length': 6, 'max_sequences': 500, 'significance': 0.01},
    'distinct_3_state': {'max_suffix_length': 8, 'max_sequences': 750, 'significance': 0.001},
    'unifilar_3_state': {'max_suffix_length': 8, 'max_sequences': 750, 'significance': 0.001},
    'distinct_4_state': {'max_suffix_length': 10, 'max_sequences': 1000, 'significance': 0.001},
    'distinct_6_state': {'max_suffix_length': 10, 'max_sequences': 1000, 'significance': 0.001},
    'seven_state_human': {'max_suffix_length': 10, 'max_sequences': 1000, 'significance': 0.001}
}

params = extraction_params.get(MACHINE, extraction_params['golden_mean'])

print(f"Extracting FSM from {MACHINE} model...")
print(f"\nCommand to run:")
print(f"python cssr_enhanced_extractor.py \\")
print(f"  --checkpoint checkpoints/{MACHINE}/best.pt \\")
print(f"  --data data/{MACHINE}/{MACHINE}/{MACHINE}.dat \\")
print(f"  --output results/{MACHINE} \\")
print(f"  --max-sequences {params['max_sequences']} \\")
print(f"  --max-suffix-length {params['max_suffix_length']} \\")
print(f"  --significance {params['significance']} \\")
print(f"  --use-information-theoretic-threshold")

print(f"\nExtraction parameters:")
print(f"• Max suffix length: {params['max_suffix_length']}")
print(f"• Max sequences: {params['max_sequences']}")
print(f"• Significance level: {params['significance']}")
print(f"• Information-theoretic threshold: ENABLED 🧠")
print(f"\nℹ️  Information-theoretic threshold:")
print(f"   Uses mutual information between neural distances and future distributions")
print(f"   Automatically finds optimal neural threshold (instead of fixed 5.0)")
print(f"   More principled approach for neural-classical fusion")

## Step 6: Generate Advanced Visualizations

Create professional visualizations for the extracted FSM using domain-specific analysis:

In [ ]:
# Generate advanced visualizations and analysis
print(f"Generating advanced visualizations for {MACHINE}...")

# Choose visualization script based on machine type
if MACHINE == "seven_state_human":
    viz_script = "analysis_viz/generate_cssr_viz_unifilar.py"
    print(f"Using unifilar-specific visualization for seven_state_human machine")
else:
    viz_script = "analysis_viz/generate_cssr_viz_unifilar.py"
    print(f"⚠️  Note: Using unifilar visualization script - may not have perfect ground truth alignment for {MACHINE}")

print(f"\nCommand to run:")
print(f"python {viz_script} \\")
print(f"  --results {NOTEBOOK_FOLDER}/results/cssr_enhanced_results.json \\")
print(f"  --output {NOTEBOOK_FOLDER}/visualizations")

print(f"\nThis will generate:")
print(f"• emissions_scatter.png: Extracted vs ground truth emission patterns")
print(f"• alignment_bars.png: L1 error per matched state")
print(f"• state_sizes.png: Usage frequency of each extracted state")
print(f"• distance_heatmap.png: Distance matrix between states")
print(f"• transition_heatmap_0.png & transition_heatmap_1.png: Transition matrices")
print(f"• summary.json: Comprehensive analysis metrics")

if MACHINE != "seven_state_human":
    print(f"\n💡 For perfect ground truth alignment, consider using seven_state_human machine")
    print(f"   Other machines will show structural analysis but ground truth comparison may be approximate")

# Execute visualization generation
!python {viz_script} \
  --results {NOTEBOOK_FOLDER}/results/cssr_enhanced_results.json \
  --output {NOTEBOOK_FOLDER}/visualizations

## Alternative: Run Everything with Single Commands

For quick experimentation, you can also run everything from the command line:

```bash
# Complete pipeline for golden_mean (example)
MACHINE="golden_mean"
NOTEBOOK_FOLDER="notebook_experiments/$MACHINE"

python pysm_generator.py --machine $MACHINE --length 80000 --output $NOTEBOOK_FOLDER/data --seed 42
python train.py --config $NOTEBOOK_FOLDER/configs/$MACHINE_config.yaml  
python cssr_enhanced_extractor.py --checkpoint $NOTEBOOK_FOLDER/checkpoints/best.pt --data $NOTEBOOK_FOLDER/data/$MACHINE/$MACHINE.dat --output $NOTEBOOK_FOLDER/results --max-sequences 500 --max-suffix-length 6 --significance 0.01 --use-information-theoretic-threshold
python analysis_viz/generate_cssr_viz_unifilar.py --results $NOTEBOOK_FOLDER/results/cssr_enhanced_results.json --output $NOTEBOOK_FOLDER/visualizations
```

## Files Generated

After running experiments, you'll have everything organized in dedicated notebook folders:

```
notebook_experiments/
└── {machine}/                    # Dedicated folder per machine
    ├── data/
    │   └── {machine}/
    │       ├── {machine}.dat          # Sequence data
    │       ├── {machine}.states       # State trajectories  
    │       ├── {machine}.machine.json # Ground truth FSM
    │       └── {machine}.meta.json    # Metadata
    ├── configs/
    │   └── {machine}_config.yaml      # Training configuration
    ├── checkpoints/
    │   ├── best.pt                    # Best trained model
    │   ├── latest.pt                  # Latest checkpoint
    │   └── training_history.json      # Training metrics
    ├── results/
    │   └── cssr_enhanced_results.json # Extracted FSM + analysis
    └── visualizations/                # Advanced visualizations
        ├── emissions_scatter.png
        ├── alignment_bars.png
        ├── state_sizes.png
        ├── distance_heatmap.png
        ├── transition_heatmap_0.png
        ├── transition_heatmap_1.png
        └── summary.json
```

## Next Steps

- Try different machines to see how complexity affects extraction
- Compare extraction results with ground truth state counts
- Experiment with extraction parameters (significance levels, suffix lengths)
- Analyze emission patterns and state separability
- Use advanced visualizations to understand FSM structure and quality
- All outputs are cleanly organized in `notebook_experiments/{machine}/` folders

**You're now running the complete Neural CSSR pipeline with organized outputs! 🎉**